# Neo4j Legal RAG Search

Notebook nay dung LangChain + OpenAI + Neo4j de retrieve, augment va generate.

In [ ]:
import json
import os
from pathlib import Path
from neo4j import GraphDatabase
from dotenv import load_dotenv

load_dotenv()

JSON_PATH = Path(r"C:/Users/Admin/Desktop/test_datn/deepseek_part2.json")
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print(f"Connected to Neo4j at {NEO4J_URI}")
print(f"JSON path: {JSON_PATH}")

In [ ]:
from legal_rag_service import LegalRAGService

service = LegalRAGService(
    json_path=JSON_PATH,
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
    openai_api_key=os.getenv("OPENAI_API_KEY"),
    model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
)

user_prompt = "Công dân Việt Nam nào câu kết với nước ngoài nhằm gây nguy hại cho độc lập, chủ quyền, thống nhất và toàn vẹn lãnh thổ của Tổ quốc thì người đó bị tội gì"

result = service.generate(user_prompt)

print("Hints tu LangChain/OpenAI:")
print(json.dumps(result["hints"], ensure_ascii=False, indent=2))
print("\nSearch terms:", result["search_terms"])
print("Prompt terms:", result["prompt_terms"])
print("Amount parsed:", result["amount_value"])

print("\nKet qua query Neo4j:")
for row in result["rows"]:
    print(json.dumps(row, ensure_ascii=False, indent=2))

print("\nGiai thich xac dinh:")
print(result["explanation"])

print("\nCau tra loi RAG cuoi cung:")
print(result["final_answer"])

In [ ]:
service.close()
driver.close()